# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
"""
Building on the ML-04 data contract for this lane (Refresh / Content Opportunity Scoring):
this section constructs an actual feature matrix from the starter CSV — log transforms for
heavy-tailed traffic counts, has_* flags for columns whose missingness is patterned by
content_type (rather than a blind fillna(0), which would silently encode content type into
the feature), and explicit categorical handling.

Note: this builds the vector on the starter CSV, fully verifiable here. The same
feature-engineering logic carries over unchanged to fact_content_daily_performance in the
warehouse — only the SQL loading step would differ.

"""

"\n\n## 1. Unit of analysis + time window\n\nOne row represents the daily search performance of a single content item, for one client,\non one calendar day, from `fact_content_daily_performance`.\n\nThis notebook uses **March 2026 (`month = '2026-03'`)**, a mid-panel month. The `_sample`\ntable contains only June 2026 (the final month) and is used only to test query mechanics —\nnever to build labels, since the final month is the natural outcome window of any\npast→future label and would leak the future into the data.\n\n"

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

fv = pd.DataFrame(index=df.index)

fv["content_id"] = df["content_id"]
fv["client_id"] = df["client_id"]

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    fv[f"log_{col}"] = np.log1p(df[col])

for col in ["content_age_days", "days_since_last_update", "ctr", "avg_position",
            "engagement_rate", "ai_traffic_pct",
            "days_with_impressions", "days_with_sessions"]:
    fv[col] = df[col]

fv["has_scroll_rate"] = df["scroll_rate"].notna().astype(int)  # blank when pageviews_90d == 0
fv["scroll_rate"] = df["scroll_rate"].fillna(0)

fv["has_word_count"] = df["word_count"].notna().astype(int)
fv["word_count_filled"] = df["word_count"].fillna(0)

fv["has_keyword_data"] = df["search_volume"].notna().astype(int)
fv["search_volume_filled"] = df["search_volume"].fillna(0)
fv["competition_filled"] = df["competition"].fillna(0)

fv["content_type"] = df["content_type"].astype("category")
fv["main_intent"] = df["main_intent"].fillna("unknown").astype("category")
fv["competition_level"] = df["competition_level"].fillna("unknown").astype("category")

print("Feature vector shape:", fv.shape)
print("\nAny nulls remaining?")
print(fv.isna().sum()[fv.isna().sum() > 0] if fv.isna().sum().sum() else "None — every column is fully filled or flagged.")
fv.head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
"""

| Feature | Meaning | Missing handling | Categorical? | Available before decision? |
|---|---|---|---|---|
| log_impressions_90d | log1p of 90-day GSC impressions | never blank | no | Yes |
| log_clicks_90d / log_sessions_90d / log_ai_sessions_90d | log1p of 90-day totals | never blank | no | Yes |
| content_age_days | days since content created | never blank | no | Yes |
| days_since_last_update | days since last edit | never blank | no | Yes |
| ctr, avg_position, engagement_rate, ai_traffic_pct | 90-day rates | never blank | no | Yes |
| has_scroll_rate / scroll_rate | scroll depth rate, whether computable | blank when pageviews_90d==0 (125 rows) — flagged then filled with 0 | no | Yes |
| days_with_impressions, days_with_sessions | activity spread | never blank | no | Yes |
| has_word_count / word_count_filled | measured or not, value | flagged, not silently zeroed | no | Yes |
| has_keyword_data / search_volume_filled / competition_filled | keyword data or not, values | flagged; 2,468 rows have none (all feedly article) | no | Yes |
| content_type | article production type | never blank | yes | Yes |
| main_intent | search intent category | filled "unknown" (2,374 rows) | yes | Yes |
| competition_level | keyword competition tier | filled "unknown" (2,610 rows) | yes | Yes |

Why flags instead of fillna(0) everywhere: word_count is missing for 28.3% of keyword
article rows but 0% of feedly/comparison rows. A plain zero-fill would make "0 words" a
proxy for content type — the model would learn content type through the back door.

"""

'\n\n## 2. Fields: feature / label / context / excluded\n\n### Features (knowable before the refresh decision)\n- `impressions` — daily search impressions, summed over the window, before the decision point\n- `clicks` — daily search clicks, summed over the window, before the decision point\n- `ctr` — computed from clicks/impressions, before the decision point\n- `avg_position` — average Search Console ranking position, before the decision point\n- `content_age_days` — static content metadata, known at any point in time\n\n### Label\n- `is_declining` — constructed by me: comparing impressions in the second half of March\n  vs. the first half of March. 1 if impressions declined, 0 otherwise. Built the same way\n  the CSV teaching slice\'s `trend_direction == "down"` label was built, just on a smaller\n  window since I only have March loaded.\n\n### Context (for joins/grouping, not features)\n- `content_id`, `client_id`, `report_date`, `month`\n\n### Excluded\n- Any raw trend/direction co

In [ ]:
print("word_count missing overall:", df["word_count"].isna().sum())
print("word_count missing by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3))

print("\nmain_intent missing:", df["main_intent"].isna().sum())
print("competition_level missing:", df["competition_level"].isna().sum())
print("search_volume missing (no keyword data):", df["search_volume"].isna().sum())
print("rows where search_volume is missing, by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(3))

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
"""
The label is is_declining_label = (trend_direction == "down"), built from trend_pct
(impressions_last_30d vs impressions_prev_30d). The attack: prove the feature vector
contains none of those columns, and prove why they'd be dangerous if it did.
"""

'\n\n## 3. Verification\n\nThe following queries verify the data contract:\n- the data grain (one row per content item per client per day)\n- row count and date range\n- missing values\n- data availability\n\n'

In [ ]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

LABEL_DERIVED = ["trend_direction", "trend_pct",
                  "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                  "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

leaked = [c for c in LABEL_DERIVED if c in fv.columns]
assert not leaked, f"LEAKAGE: these label-derived columns are in the feature vector: {leaked}"
print("Leakage test 1 (label-derived columns excluded from fv):", "PASS — none present" if not leaked else "FAIL")

# Pearson correlation UNDERSTATES this — trend_direction is a threshold/step function of
# trend_pct, not linear. The honest test: can a simple threshold reconstruct the label?
trend_pct_corr = df[["trend_pct", "is_declining_label"]].corr().iloc[0, 1]
threshold_pred = (df["trend_pct"] < -20).astype(int)
threshold_accuracy = (threshold_pred == df["is_declining_label"]).mean()
print(f"\nPearson correlation of trend_pct with is_declining_label: {trend_pct_corr:.3f} (looks weak — misleading, see below)")
print(f"Accuracy of the rule 'trend_pct < -20' at reconstructing is_declining_label: {threshold_accuracy:.4f}")
print("-> a one-line threshold on trend_pct reconstructs the label almost perfectly (99.99%).")
print("   That's the real leak: trend_pct doesn't just correlate with the label, it (almost) IS it.")

numeric_features = fv.select_dtypes(include=[np.number]).columns
numeric_features = [c for c in numeric_features if c not in ("content_id", "client_id")]
label_corrs = fv[numeric_features].join(df["is_declining_label"]).corr()["is_declining_label"].drop("is_declining_label")
print("\nCorrelation of each real feature with the label (sorted by |r|):")
print(label_corrs.sort_values(key=abs, ascending=False).round(3))
suspicious = label_corrs[label_corrs.abs() > 0.6]
print("\nAny feature with |r| > 0.6 (leakage-suspicious)?",
      f"YES: {list(suspicious.index)} — investigate before modeling" if len(suspicious) else "No — none found")

PRODUCT_FLAGS_EXCLUDED = ["provider_used", "model_used"]
in_fv = [c for c in PRODUCT_FLAGS_EXCLUDED if c in fv.columns]
print("\nLeakage test 4 (production-detail flags excluded from fv):",
      "PASS — none present" if not in_fv else f"FAIL: {in_fv}")

future_like = [c for c in fv.columns if "future" in c.lower() or "next" in c.lower()]
print("\nLeakage test 5 (no future-labeled columns):", "PASS" if not future_like else f"FAIL: {future_like}")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
"""
| Excluded field | Why |
|---|---|
| trend_direction | Is the label itself. |
| trend_pct | Builds trend_direction; a threshold on it reconstructs the label at 99.99% — reading the answer back. |
| impressions_last_30d, impressions_prev_30d | The two columns trend_pct is computed from. |
| clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d | Same 30-vs-30 window structure as the label-defining columns — excluded out of caution. |
| provider_used, model_used | Describe how content was produced, not how it's performing. Dictionary flags both "Not a model feature." |
| content_id, client_id | Pseudonymous identifiers — context for grouping/joins only, never a learned category. |

Warehouse-specific exclusions (documented in ML-04, not independently re-verified —
no HF access from this sandbox):
- GA4 columns where ga4_data_available = FALSE — zero-filled placeholders, not true zeros.
- fact_content_query_90d's *_last30-style columns, if the label window overlaps that
  table's fixed 90-day window — only *_prev30 columns would be safe.

"""

"\n\n## 4. Data limits\n\n- The warehouse is an unbalanced panel — different clients have different amounts of\n  historical data, so a raw row count isn't directly comparable across clients.\n- Some early records have Google Search Console data but not yet Google Analytics data\n  (`ga4_data_available = FALSE`) — those rows are zero-filled, not truly zero-engagement.\n- Feature and label windows must be aligned carefully. Here, the label is built only from\n  the second half of March, so features summed over the full month technically overlap the\n  label window slightly — a stricter version would compute features only from the first half.\n- External factors (algorithm updates, competitor activity, campaigns) aren't captured here.\n- These results are decision-support and directional, not definitive predictions.\n\n"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.